# NB10 — Modernized S3.1 on V5

Goal: rerun the old S3.1 idea under the **current V5 regime**, then compare scorer metrics and pure LOO against frozen V5.

Preserved from old S3.1:
- paired-family batching (positive + paired negative stay in the same train batch);
- total loss = BCE + `0.5 * paired_logistic_ranking_loss`.

Updated to current V5:
- category init `N(0, 1/sqrt(32))` with current post-MLP initialization order;
- FP32 training + FP32 validation;
- max 60 epochs, patience 10, minimum 30 epochs;
- same LR, batch size, architecture, frozen data, and validation ROC-AUC selection;
- test split is never loaded.

After training, this notebook reports AUC, FITB, mean/median margin, validation BCE, and runs pure LOO on original outfits with `n >= 4` for both frozen V5 and the modernized S3.1 checkpoint.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "exp/v5-paired-ranking-recheck"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
subprocess.run(
    ["git", "-C", str(REPO_ROOT), "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
    check=True,
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("Branch:", BRANCH)
print("Git HEAD:", HEAD)


In [ ]:
import yaml
import torch

from src.data.runtime_paths import load_runtime_paths
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.model import TypeAwarePairwiseScorer
from src.scorer.train import build_train_valid_loaders, evaluate_epoch, seed_everything
from src.scorer.paired_ranking_experiment import (
    build_paired_train_loader,
    evaluate_pure_loo_4plus,
    fit_paired_ranking_scorer,
)

CONFIG_PATH = REPO_ROOT / "configs" / "scorer_s3_1_modernized_v5.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

training = config["training"]
experiment = config["experiment"]
RANKING_WEIGHT = float(experiment["ranking_weight"])

assert RANKING_WEIGHT == 0.50
assert training["mixed_precision"] is False
assert training["max_epochs"] == 60
assert training["early_stopping_patience"] == 10
assert training["early_stopping_min_epochs"] == 30
assert training["learning_rate"] == 0.0003
assert training["seed"] == 42
assert config["model"]["pair_hidden_dim"] == 128
assert config["model"]["output_hidden_dim"] == 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Use a GPU runtime; this experiment trains in FP32."

paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)
assert provenance["git_tree_clean"] is True

loaders = build_train_valid_loaders(paths, config, num_workers=0)
train_dataset = loaders["datasets"]["train"]
valid_dataset = loaders["datasets"]["valid"]
valid_loader = loaders["valid_loader"]

assert len(train_dataset) == 30918
assert len(valid_dataset) == 2284
assert len(train_dataset.pair_families) == 15459
assert len(valid_dataset.pair_families) == 1142

print("CONFIG / DATA / PROVENANCE: PASS")
print("Ranking weight:", RANKING_WEIGHT)
print("Device:", device)


In [ ]:
# Verify that this really uses the current V5 category initialization.
seed_everything(int(training["seed"]))
probe = TypeAwarePairwiseScorer.from_config(config)
with torch.no_grad():
    category_norm_mean = float(probe.category_embedding.weight[1:].norm(dim=1).mean())

print("category init policy:", probe.category_embedding_init_policy)
print("category init std:", probe.category_embedding_init_std)
print("expected 1/sqrt(32):", 32 ** -0.5)
print("mean category norm:", category_norm_mean)

assert abs(probe.category_embedding_init_std - (32 ** -0.5)) < 1e-12
del probe


In [ ]:
RUN_DIR = Path("/content/drive/MyDrive/s3_1_modernized_v5/rank_w_0p50_seed42")
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = RUN_DIR / "best.pt"

seed_everything(int(training["seed"]))
model = TypeAwarePairwiseScorer.from_config(config).to(device)
paired_train_loader = build_paired_train_loader(train_dataset, config, num_workers=0)

if not BEST_PATH.is_file():
    fit_result = fit_paired_ranking_scorer(
        model,
        paired_train_loader,
        valid_loader,
        config=config,
        checkpoint_dir=RUN_DIR,
        provenance=provenance,
        ranking_weight=RANKING_WEIGHT,
        device=device,
    )
    print("TRAIN RESULT:")
    print(json.dumps({
        "best_epoch": fit_result["best_epoch"],
        "best_valid_roc_auc": fit_result["best_valid_roc_auc"],
        "ranking_weight": fit_result["ranking_weight"],
        "stopped_early": fit_result["stopped_early"],
    }, indent=2))
else:
    print("Existing checkpoint found; skipping retraining:", BEST_PATH)


In [ ]:
candidate_model = TypeAwarePairwiseScorer.from_config(config).to(device)
payload = load_checkpoint(
    BEST_PATH,
    model=candidate_model,
    map_location=device,
    current_provenance=provenance,
)
candidate_model.eval()

criterion = torch.nn.BCEWithLogitsLoss()
candidate_valid = evaluate_epoch(
    candidate_model,
    valid_loader,
    criterion=criterion,
    device=device,
)

baseline = {
    "roc_auc": float(experiment["baseline_valid_roc_auc"]),
    "fitb_2way": float(experiment["baseline_valid_fitb_2way"]),
    "mean_logit_margin": float(experiment["baseline_mean_logit_margin"]),
    "median_logit_margin": float(experiment["baseline_median_logit_margin"]),
}

print("\nSCORER COMPARISON")
print(f"{'metric':26s} {'V5 baseline':>14s} {'S3.1 modernized':>18s} {'delta':>12s}")
for key in ["roc_auc", "fitb_2way", "mean_logit_margin", "median_logit_margin"]:
    base = baseline[key]
    cand = float(candidate_valid[key])
    print(f"{key:26s} {base:14.6f} {cand:18.6f} {cand-base:+12.6f}")

print(f"{'valid_loss':26s} {'0.697292':>14s} {float(candidate_valid['loss']):18.6f}")
print("best_epoch:", int(payload["epoch"]))
print("sample_count:", int(candidate_valid["sample_count"]))
print("paired_family_count:", int(candidate_valid["paired_family_count"]))
print("TEST SPLIT WAS NOT LOADED.")


In [ ]:
# Pure LOO comparison on canonical original outfits n>=4.
canonical_config_path = REPO_ROOT / "configs" / "scorer_type_aware_pairwise_v1_val_auc.yaml"
with canonical_config_path.open("r", encoding="utf-8") as f:
    canonical_config = yaml.safe_load(f)

frozen_path = (
    REPO_ROOT
    / "artifacts"
    / "checkpoints"
    / "type_aware_pairwise_v1"
    / "final_val_auc_v5_seed42"
    / "best.pt"
)
assert frozen_path.is_file(), frozen_path

baseline_model = TypeAwarePairwiseScorer.from_config(canonical_config).to(device)
load_checkpoint(
    frozen_path,
    model=baseline_model,
    map_location=device,
    current_provenance=provenance,
)
baseline_model.eval()

print("Running pure LOO (n>=4) for frozen V5...")
baseline_loo = evaluate_pure_loo_4plus(baseline_model, valid_dataset, device=device)

print("Running pure LOO (n>=4) for modernized S3.1...")
candidate_loo = evaluate_pure_loo_4plus(candidate_model, valid_dataset, device=device)

print("\nPURE LOO COMPARISON — ORIGINAL n>=4")
print(f"{'metric':30s} {'V5 baseline':>14s} {'S3.1 modernized':>18s} {'delta':>12s}")
for key in ["top1_localization_accuracy", "hit_at_2", "hit_at_3"]:
    base = float(baseline_loo[key])
    cand = float(candidate_loo[key])
    print(f"{key:30s} {base:14.6f} {cand:18.6f} {cand-base:+12.6f}")

print("sample_count:", candidate_loo["sample_count"])

summary = {
    "scorer": {
        "best_epoch": int(payload["epoch"]),
        "ranking_weight": RANKING_WEIGHT,
        "validation": {k: float(v) if isinstance(v, (int, float)) else v for k, v in candidate_valid.items()},
    },
    "loo_n_ge_4": {
        "v5": {k: v for k, v in baseline_loo.items() if k != "records"},
        "s3_1_modernized": {k: v for k, v in candidate_loo.items() if k != "records"},
    },
}
SUMMARY_PATH = RUN_DIR / "s3_1_modernized_summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", SUMMARY_PATH)
